In [6]:
import sys
!{sys.executable} -m pip install faiss-cpu

  Using cached faiss_cpu-1.14.3-cp314-cp314-win_amd64.whl.metadata (7.8 kB)
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.4 MB 407.8 kB/s eta 0:00:40
   - -------------------------------------- 0.5/16.4 MB 407.8 kB/s eta 0:00:40
   - -------------------------------------- 0.5/16.4 MB 407.8 kB/s eta 0:00:40
   - -------------------------------------- 0.8/16.4 MB 368.2 kB/s eta 0:00:43
   - -------------------------------------- 0.8/16.4 MB 368.2 kB/s eta 0:00:43
   - ----------------------------------

In [1]:
# Stratified sampling 
import polars as pl

# 1. Load the cleaned data generated in Task 1
df_cleaned = pl.read_csv('../data/processed/filtered_complaints.csv')

# 2. Define stratified sampling strategy
# We want 15,000 total. We take a proportional sample from each product category.
total_sample_size = 15000
fraction = total_sample_size / len(df_cleaned)

# Perform stratified sampling
# Note: seed=42 ensures reproducibility for your professional report
df_sample = df_cleaned.sample(fraction=fraction, seed=42)

print(f"Sampling Complete.")
print(f"Original Size: {len(df_cleaned):,}")
print(f"Sample Size: {len(df_sample):,}")
print("\nRepresentation per category:")
print(df_sample['product_category'].value_counts())

Sampling Complete.
Original Size: 453,540
Sample Size: 15,000

Representation per category:
shape: (4, 2)
┌──────────────────┬───────┐
│ product_category ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ Money Transfer   ┆ 3338  │
│ Savings Account  ┆ 4571  │
│ Personal Loan    ┆ 906   │
│ Credit Card      ┆ 6185  │
└──────────────────┴───────┘


In [2]:
# Text Chunking Strategy
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the Splitter
# Choice: 500 chars with 50 overlap ensures context is preserved across chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
metadata = []

print("Starting chunking process...")

# Iterate through the sampled dataframe
for row in df_sample.to_dicts():
    narrative = str(row['cleaned_narrative'])
    
    # Split the narrative into pieces
    doc_chunks = text_splitter.split_text(narrative)
    
    for i, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        metadata.append({
            "complaint_id": row.get('Complaint ID', 'N/A'),
            "product": row['product_category'],
            "issue": row.get('Issue', 'N/A'),
            "chunk_index": i
        })

print(f"Created {len(chunks):,} chunks from {len(df_sample):,} narratives.")

c:\Users\Betty\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting chunking process...
Created 41,297 chunks from 15,000 narratives.


In [3]:
#  Initializing the Embedding Model
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model
print("Loading all-MiniLM-L6-v2 model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate Embeddings
print("Generating embeddings (this may take a few minutes)...")
embeddings = model.encode(chunks, show_progress_bar=True, batch_size=64)

# Convert to float32 for FAISS compatibility
embeddings = np.array(embeddings).astype('float32')
print(f"Embeddings shape: {embeddings.shape}")

Loading all-MiniLM-L6-v2 model...


c:\Users\Betty\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Betty\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3102.79it/s]


Generating embeddings (this may take a few minutes)...


Batches: 100%|██████████| 646/646 [16:30<00:00,  1.53s/it]


Embeddings shape: (41297, 384)


In [ ]:
#  Building the Vector Store Index (FAISS)
import faiss
import pickle
import os

# 1. Create the FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension) # Using L2 distance for similarity
index.add(embeddings)

# 2. Ensure directory exists
os.makedirs('../vector_store', exist_ok=True)

# 3. Save the Index
faiss.write_index(index, "../vector_store/complaints_index.faiss")

# 4. Save the Metadata and Chunks (so we can retrieve the text later)
with open("../vector_store/metadata.pkl", "wb") as f:
    pickle.dump({"chunks": chunks, "metadata": metadata}, f)

print("Vector store successfully persisted in 'vector_store/' directory.")

In [8]:
# Testing the Retrieval (Verification)
query = "Why are people unhappy with Credit Cards?"
query_vector = model.encode([query]).astype('float32')

# Search for the top 3 most relevant chunks
k = 3
distances, indices = index.search(query_vector, k)

print(f"Query: {query}\n")
for i, idx in enumerate(indices[0]):
    print(f"Result {i+1} (Score: {distances[0][i]:.4f}):")
    print(f"Product: {metadata[idx]['product']}")
    print(f"Text: {chunks[idx]}")
    print("-" * 30)

Query: Why are people unhappy with Credit Cards?

Result 1 (Score: 0.6827):
Product: Credit Card
Text: they only care about the that was spent using their credit card
------------------------------
Result 2 (Score: 0.7965):
Product: Credit Card
Text: consumers with less than perfect credit or no credit who are just starting to build their credit
------------------------------
Result 3 (Score: 0.8394):
Product: Credit Card
Text: individual erodes my confidence in this otherwise excellent credit card product with cash rewards and undermines my economic activities their denial keeping me stuck in a small credit limit is predatory because it results in sometimes my balance being reported as high relative to the credit limit thereby hurting my credit score and keeping me financially constrained background my consumer credit report is perfect i have no late payments no collections judgments or chargeoffs i value credit as a
------------------------------
